# 05 — Determinantes

**Objetivo:** Entender el determinante como factor de escala de volumen y como detector de singularidad de matrices.

## Contexto matemático

Para una matriz $A \in \mathbb{R}^{n\times n}$, el determinante $\det(A)$ mide cuánto escala el volumen cuando aplicamos $A$ como transformación lineal.

**Para $2\times 2$:**

$$\det\begin{pmatrix} a & b \\ c & d \end{pmatrix} = ad - bc$$

**Para $3\times 3$ (expansión por cofactores):**

$$\det(A) = a_{11}(a_{22}a_{33} - a_{23}a_{32}) - a_{12}(a_{21}a_{33} - a_{23}a_{31}) + a_{13}(a_{21}a_{32} - a_{22}a_{31})$$

In [ ]:
import os
os.chdir('/Volumes/SSD_Gabo/proyectos/growth-analytics')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(42)
print('Setup listo')

## 1 — Determinante 2×2 y significado geométrico

El valor absoluto del determinante es el **área del paralelogramo** formado por las filas (o columnas) de la matriz.

In [ ]:
# Dos vectores de atribución de canal
v1 = np.array([3.0, 1.0])
v2 = np.array([1.0, 2.5])

A2 = np.vstack([v1, v2])
det_2x2 = np.linalg.det(A2)
manual_2x2 = v1[0]*v2[1] - v1[1]*v2[0]

print(f'v1 = {v1},  v2 = {v2}')
print(f'det(A) = v1[0]*v2[1] - v1[1]*v2[0] = {manual_2x2:.4f}')
print(f'np.linalg.det(A) = {det_2x2:.4f}')
print(f'Área del paralelogramo = |det| = {abs(det_2x2):.4f}')

## 2 — Propiedades del determinante

$$\det(AB) = \det(A)\det(B)$$

$$\det(A^T) = \det(A)$$

$$\det(cA) = c^n \det(A) \quad \text{(para } A \in \mathbb{R}^{n\times n}\text{)}$$

$$\det(A) = 0 \iff A \text{ es singular (no invertible)}$$

In [ ]:
np.random.seed(42)
M = np.random.randint(1, 6, (3, 3)).astype(float)
N = np.random.randint(1, 6, (3, 3)).astype(float)

print(f'det(MN) = {np.linalg.det(M@N):.4f}')
print(f'det(M)*det(N) = {np.linalg.det(M)*np.linalg.det(N):.4f}')
print(f'¿Iguales? {np.isclose(np.linalg.det(M@N), np.linalg.det(M)*np.linalg.det(N))}')

print(f'\ndet(M^T) = {np.linalg.det(M.T):.4f}')
print(f'det(M)   = {np.linalg.det(M):.4f}')

c, n = 2.0, 3
print(f'\ndet(2M) = {np.linalg.det(c*M):.4f}')
print(f'2^3 * det(M) = {c**n * np.linalg.det(M):.4f}')

## 3 — Matriz singular: det = 0 y multicolinealidad

En analytics, una matriz de canales singular significa que alguna combinación lineal de canales es redundante: los vectores de atribución no son independientes.

Un **determinante cercano a cero** (aunque no exactamente cero) indica **multicolinealidad**: los canales están tan correlacionados que el sistema no puede distinguir sus efectos individuales.

In [ ]:
# Canales perfectamente colineales
A_sing = np.array([
    [1.0, 2.0, 3.0],
    [2.0, 4.0, 6.0],   # = 2 × fila 0 → dependiente
    [1.0, 1.0, 2.0],
])
print('det singular:', np.linalg.det(A_sing))

# Canales casi colineales (correlación alta)
canal_paid_search = np.array([100, 200, 150, 300, 250])
canal_display     = canal_paid_search * 0.95 + np.random.normal(0, 3, 5)   # casi idéntico
canal_email       = np.array([80, 90, 110, 130, 100])

A_attr = np.column_stack([canal_paid_search, canal_display, canal_email]).astype(float)
print(f'\ndet(A_atribución) = {np.linalg.det(A_attr):.4f}')
print(f'Número de condición = {np.linalg.cond(A_attr):.1f}')
print('→ Cond muy alto = multicolinealidad severa entre Paid Search y Display')

## 4 — Visualización: área del paralelogramo = |det|

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
casos = [
    (np.array([2., 0.]), np.array([0., 2.]), 'Ortogonales\n(área máxima para ||v||)'),
    (np.array([3., 1.]), np.array([1., 2.5]), 'Ángulo intermedio'),
    (np.array([2., 1.]), np.array([4., 2.]), 'Colineales\n(área = 0, det = 0)'),
]

for ax, (u, v, title) in zip(axes, casos):
    area = abs(u[0]*v[1] - u[1]*v[0])
    # Paralelogramo
    xs = [0, u[0], u[0]+v[0], v[0], 0]
    ys = [0, u[1], u[1]+v[1], v[1], 0]
    ax.fill(xs, ys, alpha=0.25, color='#4361ee')
    ax.plot(xs, ys, color='#4361ee', lw=1.5)
    ax.annotate('', xy=u, xytext=(0,0), arrowprops=dict(arrowstyle='->', color='#4361ee', lw=2))
    ax.annotate('', xy=v, xytext=(0,0), arrowprops=dict(arrowstyle='->', color='#f72585', lw=2))
    ax.text(u[0]/2, u[1]/2+0.1, 'u', color='#4361ee', fontweight='bold')
    ax.text(v[0]/2+0.1, v[1]/2, 'v', color='#f72585', fontweight='bold')
    ax.set_title(f'{title}\nÁrea = |det| = {area:.2f}', fontsize=10)
    lim = max(u[0]+v[0], u[1]+v[1]) + 0.5
    ax.set_xlim(-0.5, lim); ax.set_ylim(-0.5, lim)
    ax.set_aspect('equal')
    ax.axhline(0, color='#ccc', lw=0.5); ax.axvline(0, color='#ccc', lw=0.5)

plt.suptitle('Determinante = Área del paralelogramo (con signo)', fontweight='bold')
plt.tight_layout()
plt.show()

## Resumen

| Concepto | Fórmula/Descripción | NumPy |
|---|---|---|
| Determinante 2×2 | $ad - bc$ | `np.linalg.det(A)` |
| Significado geométrico | Área/volumen firmado | — |
| det(AB) = det(A)det(B) | Multiplicatividad | — |
| det = 0 | Matriz singular, sin inversa | — |
| Multicolinealidad | det ≈ 0 en datos reales | `np.linalg.cond(A)` |

**Siguiente:** `06_inverses.ipynb`